## A6a - CME catalog ingestion and feature grid

This notebook ingests the SOHO/LASCO CME catalog and converts the event list into a fixed 3h time grid. The output is used as solar-origin features for the extension experiments.

### Data source and download
The CME catalog is the CDAW universal catalog file:
https://cdaw.gsfc.nasa.gov/CME_list/UNIVERSAL_ver2/text_ver/univ_all.txt

Primary CDAW links:
- https://cdaw.gsfc.nasa.gov/
- https://cdaw.gsfc.nasa.gov/CME_list/

### Catalog fields (from 1996 to 2025)
The file contains the following columns:
Date, Time, Central PA, Width, Linear Speed, 2nd order speed (initial, final, 20R), Accel, Mass, Kinetic Energy, MPA, Remarks


In [10]:
# Load local SOHO/LASCO CME catalog (univ_all.txt)
from pathlib import Path
import pandas as pd

# --- Config ---
CME_PATH = Path('../../Data/univ_all.txt')
START_DATE = '2010-01-01'
END_DATE = '2026-01-27'  # inclusive-ish, use < next day when filtering
CADENCE = '3h'
OUT_PATH = Path('../../Data/processed/solar_origin_cme_features.parquet')

if not CME_PATH.exists():
    raise FileNotFoundError(f'Cannot find {CME_PATH}')

rows = []
with CME_PATH.open() as f:
    for line in f:
        line = line.rstrip()
        if not line or not line[0].isdigit():
            continue
        parts = line.split()
        if len(parts) < 12:
            continue
        head = parts[:12]
        remarks = ' '.join(parts[12:]) if len(parts) > 12 else ''
        rows.append(head + [remarks])

cols = [
    'date','time','cpa','width','speed_lin',
    'speed2_init','speed2_final','speed2_20r',
    'accel','mass','kinetic','mpa','remarks'
]
df = pd.DataFrame(rows, columns=cols)

num_cols = ['cpa','width','speed_lin','speed2_init','speed2_final','speed2_20r','accel','mass','kinetic','mpa']
for col in num_cols:
    df[col] = (df[col]
               .str.replace('*','', regex=False)
               .replace({'-------': None}))
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['timestamp_utc'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce', utc=True)
df = df.dropna(subset=['timestamp_utc']).reset_index(drop=True)

# Filter to target window
start_ts = pd.Timestamp(START_DATE, tz='UTC')
end_ts = pd.Timestamp(END_DATE, tz='UTC') + pd.Timedelta(days=1)
df = df[(df['timestamp_utc'] >= start_ts) & (df['timestamp_utc'] < end_ts)].copy()

print(f'Coverage: {df.timestamp_utc.min()} to {df.timestamp_utc.max()}, {len(df)} events')

# Build fixed cadence grid
start_grid = df['timestamp_utc'].min().floor(CADENCE)
end_grid = df['timestamp_utc'].max().ceil(CADENCE)
idx = pd.date_range(start_grid, end_grid, freq=CADENCE, tz='UTC')

# Event-level helpers
is_halo = df['width'].ge(120)

# Bin rule: assign each CME to the first grid cell AFTER event start (ceil)
df['bin_time'] = df['timestamp_utc'].dt.ceil(CADENCE)

# Aggregate to cadence
agg = (df.groupby('bin_time')
       .agg({
            'speed_lin': ['mean', 'max'],
            'width': ['mean', 'max'],
            'cpa': 'mean',
            'mpa': 'mean',
        })
      )
agg.columns = ['cme_speed_mean','cme_speed_max','cme_width_mean','cme_width_max','cme_cpa_mean','cme_mpa_mean']

# Counts
counts = df.groupby('bin_time').size().rename('cme_count')
halo_counts = (df.assign(is_halo=is_halo)
               .groupby('bin_time')['is_halo']
               .sum()
               .rename('cme_halo_count'))

# Merge to full grid
out = pd.DataFrame(index=idx)
out = out.join(agg).join(counts).join(halo_counts)

# Fill missing bins with zeros for counts; keep means as NaN, then fill with 0 if desired
out['cme_count'] = out['cme_count'].fillna(0).astype(int)
out['cme_halo_count'] = out['cme_halo_count'].fillna(0).astype(int)

# Optional: fill mean/max with 0 when no events
out = out.fillna(0)

out = out.reset_index().rename(columns={'index': 'timestamp_utc'})
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(OUT_PATH, index=False)

print(f'Saved {len(out)} rows -> {OUT_PATH}')
print(out.head())




Coverage: 2010-01-01 06:54:03+00:00 to 2025-11-30 20:00:05+00:00, 24416 events
Saved 46502 rows -> ../../Data/processed/solar_origin_cme_features.parquet
              timestamp_utc  cme_speed_mean  cme_speed_max  cme_width_mean  \
0 2010-01-01 06:00:00+00:00             0.0            0.0             0.0   
1 2010-01-01 09:00:00+00:00           239.0          330.0            40.5   
2 2010-01-01 12:00:00+00:00             0.0            0.0             0.0   
3 2010-01-01 15:00:00+00:00             0.0            0.0             0.0   
4 2010-01-01 18:00:00+00:00           729.0          729.0            21.0   

   cme_width_max  cme_cpa_mean  cme_mpa_mean  cme_count  cme_halo_count  
0            0.0           0.0           0.0          0               0  
1           75.0         301.0         304.0          2               0  
2            0.0           0.0           0.0          0               0  
3            0.0           0.0           0.0          0               0  
4      